samo temperatura 1h csv za sve gradove

In [ ]:
import pandas as pd

file = "00 Bihac sat T.xlsx"
city = "Bihac"

df = pd.read_excel(file, sheet_name=0)

# clean columns (just in case)
df.columns = df.columns.astype(str).str.strip()

# rename first 3 columns
df = df.rename(columns={
    df.columns[0]: "year",
    df.columns[1]: "month",
    df.columns[2]: "hour"
})

# melt ALL day columns (they are already correct: 1–31)
long_df = df.melt(
    id_vars=["year", "month", "hour"],
    var_name="day",
    value_name="temperature"
)

# convert types safely
long_df["day"] = pd.to_numeric(long_df["day"], errors="coerce")
long_df = long_df.dropna(subset=["day", "temperature"])

long_df["day"] = long_df["day"].astype(int)

# FIX hour (your format is "00:00", "01:00")
long_df["hour"] = long_df["hour"].str[:2].astype(int)

# build datetime
long_df["datetime"] = pd.to_datetime(
    dict(
        year=long_df["year"],
        month=long_df["month"],
        day=long_df["day"],
        hour=long_df["hour"]
    ),
    errors="coerce"
)

long_df = long_df.dropna(subset=["datetime"])

long_df["city"] = city

final_df = long_df[["datetime", "temperature", "city"]]
final_df = final_df.sort_values("datetime").reset_index(drop=True)

print(final_df.head())
print(final_df.shape)

             datetime  temperature   city
0 2020-01-01 00:00:00         -1.9  Bihac
1 2020-01-01 01:00:00         -2.4  Bihac
2 2020-01-01 02:00:00         -3.0  Bihac
3 2020-01-01 03:00:00         -3.2  Bihac
4 2020-01-01 04:00:00         -2.8  Bihac
(38652, 3)


In [3]:
import pandas as pd
from pathlib import Path

# Colab default working directory
folder = Path("/content")

files = list(folder.glob("*.xlsx"))

print("FOUND FILES:", len(files))
print(files)

all_cities = []

for file in files:

    parts = file.stem.split()
    city = parts[1]  # "00 Bihac sat T" -> "Bihac"

    df = pd.read_excel(file, sheet_name=0)
    df.columns = df.columns.astype(str).str.strip()

    df = df.rename(columns={
        df.columns[0]: "year",
        df.columns[1]: "month",
        df.columns[2]: "hour"
    })

    long_df = df.melt(
        id_vars=["year", "month", "hour"],
        var_name="day",
        value_name="temperature"
    )

    long_df["day"] = pd.to_numeric(long_df["day"], errors="coerce")
    long_df = long_df.dropna(subset=["day", "temperature"])
    long_df["day"] = long_df["day"].astype(int)

    long_df["hour"] = long_df["hour"]

    final = long_df[[
        "year", "month", "day", "hour", "temperature"
    ]].copy()

    final["city"] = city

    all_cities.append(final)

if not all_cities:
    raise ValueError("No Excel files found in /content")

full_df = pd.concat(all_cities, ignore_index=True)

print(full_df.head())
print(full_df.shape)

FOUND FILES: 9
[PosixPath('/content/00 Bugojno sat T.xlsx'), PosixPath('/content/00 Tuzla sat T.xlsx'), PosixPath('/content/00 Gradacac sat T.xlsx'), PosixPath('/content/00 Zenica sat T.xlsx'), PosixPath('/content/00 Sanski Most sat T.xlsx'), PosixPath('/content/00 Bihac sat T.xlsx'), PosixPath('/content/00 Livno sat T.xlsx'), PosixPath('/content/00 Sarajevo sat T.xlsx'), PosixPath('/content/00 Mostar sat T.xlsx')]
   year  month  day   hour  temperature     city
0  2020      1    1  00:00         -5.3  Bugojno
1  2020      1    1  01:00         -5.9  Bugojno
2  2020      1    1  02:00         -6.4  Bugojno
3  2020      1    1  03:00         -7.0  Bugojno
4  2020      1    1  04:00         -7.7  Bugojno
(316183, 6)


In [4]:
full_df.to_csv("fhz_temperature_1h.csv", index=False)

print("Saved to /content/all_cities_temperature.csv")

Saved to /content/all_cities_temperature.csv
